# Training classifiers with MultiTrain

In this walkthrough, we will train MultiTrain's complete classifier catalog on the Titanic dataset included with the repository. The point is to see the measurements returned by every model, so we will keep the full results table instead of asking MultiTrain to select a model for us.

## Import MultiTrain and load the dataset

I like printing the installed version at the beginning of an example. It makes the saved notebook output useful later because we can see exactly which MultiTrain release produced the results.

In [1]:
from pathlib import Path

import pandas as pd

import MultiTrain
from MultiTrain import MultiClassifier

print(f"MultiTrain version: {MultiTrain.__version__}")

dataset_path = Path("examples/datasets/train.csv")
df = pd.read_csv(dataset_path)
df.head()

MultiTrain version: 1.2.0


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Take a quick look at the data

`Survived` is the target. `Age` and `Embarked` contain missing values, while several columns contain text that must either be encoded or removed before the models can use them.

In [2]:
print(f"Dataset shape: {df.shape}")
print()
print("Target distribution:")
print(df["Survived"].value_counts().sort_index())
print()
print("Columns containing missing values:")
print(df.isna().sum()[df.isna().any()])

Dataset shape: (891, 12)

Target distribution:
Survived
0    549
1    342
Name: count, dtype: int64

Columns containing missing values:
Age         177
Cabin       687
Embarked      2
dtype: int64


## Configure MultiTrain

Each estimator receives one CPU thread and MultiTrain may train two different estimators at the same time. `custom_models` is left as `None`, which means this run includes every built-in classifier. GPU training is intentionally left off so the notebook runs on an ordinary laptop and in CI.

In [3]:
train = MultiClassifier(
    n_jobs=1,
    model_workers=2,
    random_state=42,
    max_iter=300,
)

## Prepare the train and test sets

The passenger name, ticket, cabin, and identifier are high-cardinality identifiers rather than useful columns for this introductory example, so we drop them. MultiTrain fills the missing values after splitting the data and learns categorical encodings from the training partition only.

In [4]:
classification_split = train.split(
    data=df,
    target="Survived",
    test_size=0.2,
    random_state=42,
    auto_cat_encode=True,
    fix_nan_custom={"Age": "interpolate", "Embarked": "ffill"},
    drop=["PassengerId", "Name", "Ticket", "Cabin"],
)

X_train, X_test, y_train, y_test = classification_split
print(f"Training features: {X_train.shape}")
print(f"Test features: {X_test.shape}")
X_train.head()

Training features: (712, 7)
Test features: (179, 7)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,0,45.5,0,0,28.5000,0
733,2,0,23.0,0,0,13.0000,0
382,3,0,32.0,0,0,7.9250,0
704,3,0,26.0,1,0,7.8542,0
813,3,1,6.0,4,2,31.2750,0


## Train and measure every classifier

`show_train_score=True` places the training and test measurements next to each other. This makes it easier to inspect generalization gaps. Sorting by accuracy only changes the order of the table; every model is still present.

In [5]:
pd.set_option("display.max_rows", None)

classification_results = train.fit(
    datasplits=classification_split,
    show_train_score=True,
    sort="accuracy",
)

classification_results

Training Models:   0%|          | 0/27 [00:00<?, ?it/s]

,accuracy,precision_train,precision,recall_train,recall,balanced_accuracy_train,balanced_accuracy,accuracy_train,f1_train,f1,roc_auc_train,roc_auc,Time
AdaBoostClassifier,0.815642,0.786561,0.797101,0.742537,0.743243,0.810458,0.804955,0.827247,0.763916,0.769231,0.880589,0.867632,583.98ms
NuSVC,0.815642,0.811159,0.80597,0.705224,0.72973,0.803062,0.80296,0.827247,0.754491,0.765957,0.853654,0.834234,68.57ms
SVC,0.815642,0.862559,0.825397,0.679104,0.702703,0.806895,0.79897,0.838483,0.759916,0.759124,0.878437,0.859331,61.24ms
GradientBoostingClassifier,0.810056,0.948837,0.8125,0.761194,0.702703,0.86821,0.794208,0.894663,0.84472,0.753623,0.951333,0.892728,125.54ms
CatBoostClassifier,0.804469,0.957547,0.819672,0.757463,0.675676,0.868596,0.785457,0.896067,0.845833,0.740741,0.951047,0.893436,552.09ms
ExtraTreesClassifier,0.804469,1.0,0.753247,0.962687,0.783784,0.981343,0.801416,0.985955,0.980989,0.768212,0.999563,0.856306,154.21ms
MLPClassifier,0.804469,0.809322,0.791045,0.712687,0.716216,0.805668,0.791441,0.828652,0.757937,0.751773,0.871445,0.890862,112.16ms
LGBMClassifier,0.793296,0.992278,0.728395,0.958955,0.797297,0.977225,0.793887,0.981742,0.975332,0.76129,0.999252,0.879022,170.91ms
HistGradientBoostingClassifier,0.793296,0.996139,0.734177,0.962687,0.783784,0.980217,0.791892,0.984551,0.979127,0.75817,0.999433,0.878378,516.81ms
RandomForestClassifier,0.793296,0.992366,0.753425,0.970149,0.743243,0.982822,0.785907,0.985955,0.981132,0.748299,0.99942,0.875225,179.63ms


## Confirm the run

The final check counts the returned models and identifies models for which every test metric is missing. An individual missing metric does not necessarily mean fitting failed—for example, some estimators cannot provide the score needed for ROC AUC.

In [6]:
classification_metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "balanced_accuracy",
]
failed_classifiers = classification_results[
    classification_metrics
].isna().all(axis=1)

print(f"Models returned: {len(classification_results)}")
print(f"Models with every test metric missing: {failed_classifiers.sum()}")
if failed_classifiers.any():
    print(classification_results.index[failed_classifiers].tolist())

Models returned: 27
Models with every test metric missing: 0
